In [1]:
%cd /Users/anuraagshankar/Projects/upf/netsim/WiPySim

/Users/anuraagshankar/Projects/upf/netsim/WiPySim


In [2]:
import sys
import os

# Add project root to path if not already there
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from __future__ import annotations
from typing import Any, Tuple

import ray
from ray.rllib.algorithms.ppo import PPOConfig

from src.envs.rllib_multi_ap import MultiAPEnv

In [4]:
env = MultiAPEnv()

In [5]:
obs_map, info_map = env.reset()
step_idx = 0
print(f"Reset: {list(obs_map.keys())} agents ready")

Reset: ['ap_1'] agents ready


In [6]:
while True:
    # Sample a random action for each agent that is currently ready
    actions = {}
    for agent_key in obs_map.keys():
        act_space = env.get_action_space(agent_key)
        actions[agent_key] = int(act_space.sample())

    obs_map, rewards_map, terminateds_map, truncateds_map, infos_map = env.step(actions)
    step_idx += 1

    # Print minimal progress
    if step_idx % 10 == 0:
        print(f"Step {step_idx}: rewards snapshot: {rewards_map}")

    if terminateds_map.get("__all__", False) or truncateds_map.get("__all__", False):
        print("Episode finished.")
        break

Step 10: rewards snapshot: {'ap_4': np.float64(-826.0)}
Step 20: rewards snapshot: {'ap_1': np.float64(-8734.0)}
Step 30: rewards snapshot: {'ap_1': np.float64(-10000.0)}
Step 40: rewards snapshot: {'ap_4': np.float64(-1170.0)}
Step 50: rewards snapshot: {'ap_4': np.float64(-7371.0)}
Step 60: rewards snapshot: {'ap_4': np.float64(-627.0)}
Step 70: rewards snapshot: {'ap_4': np.float64(-9201.0)}
Step 80: rewards snapshot: {'ap_4': np.float64(-411.0)}
Step 90: rewards snapshot: {'ap_4': np.float64(-2808.0)}
Step 100: rewards snapshot: {'ap_4': np.float64(-9797.0)}
Step 110: rewards snapshot: {'ap_4': np.float64(-2500.0)}
Step 120: rewards snapshot: {'ap_1': np.float64(-9978.0)}
Step 130: rewards snapshot: {'ap_4': np.float64(-1182.0)}
Step 140: rewards snapshot: {'ap_4': np.float64(-6592.0)}
Step 150: rewards snapshot: {'ap_1': np.float64(-5325.0)}
Step 160: rewards snapshot: {'ap_4': np.float64(-2639.0)}
Step 170: rewards snapshot: {'ap_4': np.float64(-1088.0)}
Step 180: rewards snapsho

In [7]:
SHARED_POLICY_ID = "shared_policy"

# -----------------------
# Training configuration
# -----------------------
DEFAULT_ITERS: int = 50
NUM_WORKERS: int = 0
NUM_GPUS: int = 0
LR: float = 5e-4
GAMMA: float = 0.99
TRAIN_BATCH_SIZE: int = 4000
SGD_MINIBATCH_SIZE: int = 128
NUM_SGD_ITER: int = 10
FCNET_HIDDENS: Tuple[int, int] = (128, 128)
CHECKPOINT_EVERY: int = 10  # set to 0 to disable periodic checkpoints

In [8]:
def build_ppo_config(
    num_workers: int,
    num_gpus: int,
    lr: float,
    gamma: float,
    train_batch_size: int,
    sgd_minibatch_size: int,
    num_sgd_iter: int,
    fcnet_hiddens: Tuple[int, int],
) -> PPOConfig:
    # Instantiate a temporary env instance to infer spaces for the shared policy
    temp_env = MultiAPEnv()
    # Multi-agent env exposes per-agent spaces; fetch from one example agent if available
    try:
        obs_map, _ = temp_env.reset()
        example_agent = next(iter(obs_map.keys()))
        obs_space = temp_env.get_observation_space(example_agent)
        act_space = temp_env.get_action_space(example_agent)
    except (StopIteration, AttributeError, TypeError, KeyError):
        # Fallback: methods ignore agent_id and return shared spaces
        obs_space = temp_env.get_observation_space(None)
        act_space = temp_env.get_action_space(None)

    config = (
        PPOConfig()
        .environment(MultiAPEnv)
        .multi_agent(
            policies={
                SHARED_POLICY_ID: (
                    None,  # default PPO policy class
                    obs_space,
                    act_space,
                    {},
                )
            },
            # Use flexible signature to handle different RLlib versions
            policy_mapping_fn=lambda agent_id, *args, **kwargs: SHARED_POLICY_ID,
            policies_to_train=[SHARED_POLICY_ID],
        )
        .framework("torch")
        .resources(num_gpus=num_gpus)
        .env_runners(num_env_runners=num_workers)
        .training(
            lr=lr,
            gamma=gamma,
            train_batch_size=train_batch_size,
            minibatch_size=sgd_minibatch_size,
            # num_sgd_iter=num_sgd_iter,
            model={
                "fcnet_hiddens": list(fcnet_hiddens),
                "fcnet_activation": "tanh",
            },
            clip_param=0.2,
            vf_clip_param=10.0,
            # Add entropy coefficient for exploration
            entropy_coeff=0.01,
        )
        .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
    )
    return config

In [9]:
runtime_env = {
    "py_modules": [os.path.abspath(os.getcwd())]
}

ray.init(
    # num_gpus=1, 
    runtime_env=runtime_env
)

2025-10-06 17:14:21,250	INFO worker.py:1951 -- Started a local Ray instance.
2025-10-06 17:14:21,352	INFO packaging.py:588 -- Creating a file package for local module '/Users/anuraagshankar/Projects/upf/netsim/WiPySim'.
2025-10-06 17:14:21,358	WARNING packaging.py:430 -- File /Users/anuraagshankar/Projects/upf/netsim/WiPySim/tests/ws_traces/tshark_processed_traffic.tsv is very large (94.48MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['/Users/anuraagshankar/Projects/upf/netsim/WiPySim/tests/ws_traces/tshark_processed_traffic.tsv']})`
2025-10-06 17:14:21,464	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_9208dc4ebe82a6bc.zip' (104.74MiB) to Ray cluster...
2025-10-06 17:14:21,619	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_9208dc4ebe82a6bc.zip'.


Python version:,3.12.11
Ray version:,2.49.0


In [10]:
# Build and train PPO
config = build_ppo_config(
    num_workers=NUM_WORKERS,
    num_gpus=NUM_GPUS,
    lr=LR,
    gamma=GAMMA,
    train_batch_size=TRAIN_BATCH_SIZE,
    sgd_minibatch_size=SGD_MINIBATCH_SIZE,
    num_sgd_iter=NUM_SGD_ITER,
    fcnet_hiddens=FCNET_HIDDENS,
)

In [11]:
algo = config.build_algo()

/Users/anuraagshankar/Projects/upf/netsim/netsim/lib/python3.12/site-packages/ray/rllib/algorithms/algorithm.py:520: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/anuraagshankar/Projects/upf/netsim/netsim/lib/python3.12/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/Users/anuraagshankar/Projects/upf/netsim/netsim/lib/python3.12/site-packages/ray/tune/logger/unifie

In [12]:
last_checkpoint = None

In [12]:
result = algo.train()

2025-10-06 17:15:04,783	WARNING deprecation.py:50 -- DeprecationWarning: `_get_slice_indices` has been deprecated. This will raise an error in the future!


In [13]:
for i in range(1, DEFAULT_ITERS + 1):
    result = algo.train()
    # Try new-style result keys first, then fallback
    ep_reward_mean = (
        (result.get("env_runners") or {}).get("episode_reward_mean")
        if isinstance(result, dict)
        else None
    )
    if ep_reward_mean is None:
        ep_reward_mean = result.get("episode_reward_mean")
    train_timesteps = result.get("timesteps_total")
    print(
        f"Iter {i:04d} | mean_reward={ep_reward_mean} | timesteps={train_timesteps}"
    )

    if CHECKPOINT_EVERY and i % CHECKPOINT_EVERY == 0:
        saved = algo.save()
        last_checkpoint = getattr(saved, "checkpoint_path", saved)
        print(f"Saved checkpoint: {last_checkpoint}")

AttributeError: 'EnvContext' object has no attribute 'SEED'

In [ ]:
if last_checkpoint is None:
    saved = algo.save()
    last_checkpoint = getattr(saved, "checkpoint_path", saved)
    print(f"Saved final checkpoint: {last_checkpoint}")